# **OOP Project Implementation**

**DS212 — Object-Oriented Programming**

**Week 16:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Implement the OOP project using appropriate class relationships, exception handling, testing, and version control.

With an approved design, implementation begins. This lesson describes **how** to build an object-oriented project steadily and safely: in small steps, with tests, with clear error handling, and with every change recorded in Git.

```{mermaid}
flowchart LR
    P["Pick a small task<br/>(one user story)"] --> B["Branch"]
    B --> T["Write / update tests"]
    T --> C["Implement classes"]
    C --> R{"Tests pass?"}
    R -- no --> D["Debug"]
    D --> C
    R -- yes --> M["Commit, review, merge"]
    M --> P
```

## **Class Implementation and Collaboration**

### **Start from skeletons**

Translate the class diagram into **skeletons** first: class names, constructors, and method signatures with docstrings. This exposes design gaps before any detailed code is written.

```python
class Canteen:
    '''Accepts orders and produces reports.'''

    def __init__(self, menu):
        ...

    def place_order(self, customer, items):
        '''Create an Order from {code: quantity}; raise if an item is unknown.'''
        raise NotImplementedError
```

### **Build vertical slices**

Instead of finishing all classes "horizontally" (every attribute of every class first), implement **one user story end to end** — for example FR-2 *place an order* — through every class it touches. Each slice gives you something that **works** and can be demonstrated.

### **Collaboration through composition and dependency injection**

Pass collaborators **into** objects rather than creating them deep inside. This keeps classes loosely coupled and easy to test:

```python
# tightly coupled: Canteen always builds its own Menu from a file
class Canteen:
    def __init__(self):
        self.menu = Menu.from_csv("menu.csv")

# loosely coupled: any Menu can be supplied — a real one, or a small one in tests
class Canteen:
    def __init__(self, menu):
        self.menu = menu
```

The following slice implements *place an order* for CampusEats:

In [ ]:
from enum import Enum


class CampusEatsError(Exception):
    '''Base class for application errors.'''


class UnknownItemError(CampusEatsError):
    def __init__(self, code):
        self.code = code
        super().__init__(f"Unknown menu item {code!r}.")


class ItemUnavailableError(CampusEatsError):
    pass


class OrderStatus(Enum):
    PLACED = 1
    PREPARING = 2
    READY = 3
    CLAIMED = 4


class MenuItem:
    def __init__(self, code, name, price, available=True):
        if price <= 0:
            raise ValueError("Price must be positive.")
        self.code, self.name, self.price, self.available = code, name, price, available


class Menu:
    def __init__(self, items=()):
        self._items = {item.code: item for item in items}

    def get(self, code):
        try:
            return self._items[code]
        except KeyError:
            raise UnknownItemError(code) from None

    def available(self):
        return [i for i in self._items.values() if i.available]


class Order:
    def __init__(self, order_id, customer):
        self.order_id, self.customer = order_id, customer
        self._lines = []
        self._status = OrderStatus.PLACED

    @property
    def status(self):
        return self._status

    @property
    def lines(self):
        return tuple(self._lines)

    def add_line(self, item, quantity):
        if not item.available:
            raise ItemUnavailableError(f"{item.name} is sold out.")
        if quantity < 1:
            raise ValueError("Quantity must be at least 1.")
        self._lines.append((item, quantity))

    def total(self):
        return sum(item.price * qty for item, qty in self._lines)

    def advance(self):
        if self._status is OrderStatus.CLAIMED:
            raise CampusEatsError(f"Order #{self.order_id} was already claimed.")
        self._status = OrderStatus(self._status.value + 1)


class Canteen:
    def __init__(self, menu):                # dependency injection
        self._menu = menu
        self._orders = []

    def place_order(self, customer, items):
        if not items:
            raise ValueError("An order needs at least one item.")
        order = Order(len(self._orders) + 1, customer)
        for code, qty in items.items():
            order.add_line(self._menu.get(code), qty)
        self._orders.append(order)            # only stored if every line was valid
        return order

    def queue(self):
        return [o for o in self._orders if o.status is not OrderStatus.CLAIMED]

In [ ]:
menu = Menu([
    MenuItem("RICE", "Rice", 15),
    MenuItem("ADOBO", "Chicken adobo", 80),
    MenuItem("HALO", "Halo-halo", 65, available=False),
])
canteen = Canteen(menu)

order = canteen.place_order("Ana", {"RICE": 2, "ADOBO": 1})
print(f"Order #{order.order_id}: ₱{order.total():.2f} ({order.status.name})")

## **Exception Handling and Validation**

Use a consistent strategy across the project:

1. **Validate at boundaries.** Constructors and public methods check their inputs (`MenuItem` rejects non-positive prices).
2. **Raise domain exceptions** from the model classes (`UnknownItemError`, `ItemUnavailableError`), all inheriting from one base (`CampusEatsError`).
3. **Keep objects consistent.** `place_order()` stores the order only after **all** lines succeed, so a failure never leaves a half-built order.
4. **Handle at the top.** The user interface catches exceptions and shows friendly messages. Model classes never `print()` errors themselves.

In [ ]:
def ui_place_order(canteen, customer, items):
    '''The interface layer: the only place that talks to the user.'''
    try:
        order = canteen.place_order(customer, items)
    except UnknownItemError as error:
        return f"Sorry, we don't have {error.code}."
    except ItemUnavailableError as error:
        return f"Sorry: {error}"
    except (CampusEatsError, ValueError) as error:
        return f"Could not place order: {error}"
    return f"Order #{order.order_id} placed — total ₱{order.total():.2f}"


print(ui_place_order(canteen, "Ben", {"ADOBO": 1}))
print(ui_place_order(canteen, "Ben", {"PIZZA": 1}))
print(ui_place_order(canteen, "Ben", {"HALO": 1}))
print(ui_place_order(canteen, "Ben", {}))
print("Orders stored:", len(canteen.queue()))

## **Testing During Implementation**

Write tests **alongside** each class — not at the end. Each acceptance criterion from the proposal becomes at least one test.

In [ ]:
import sys
import unittest


class TestCanteen(unittest.TestCase):
    def setUp(self):
        self.canteen = Canteen(Menu([
            MenuItem("RICE", "Rice", 15),
            MenuItem("ADOBO", "Chicken adobo", 80),
            MenuItem("HALO", "Halo-halo", 65, available=False),
        ]))

    def test_total_is_sum_of_lines(self):                       # FR-2
        order = self.canteen.place_order("Ana", {"RICE": 2, "ADOBO": 1})
        self.assertEqual(order.total(), 110)

    def test_unavailable_item_rejected(self):                   # FR-2
        with self.assertRaises(ItemUnavailableError):
            self.canteen.place_order("Ana", {"HALO": 1})

    def test_failed_order_is_not_stored(self):                  # consistency
        with self.assertRaises(UnknownItemError):
            self.canteen.place_order("Ana", {"RICE": 1, "PIZZA": 1})
        self.assertEqual(self.canteen.queue(), [])

    def test_status_moves_forward_only(self):                   # FR-3
        order = self.canteen.place_order("Ana", {"RICE": 1})
        for _ in range(3):
            order.advance()
        self.assertIs(order.status, OrderStatus.CLAIMED)
        with self.assertRaises(CampusEatsError):
            order.advance()


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCanteen)
unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(suite);

## **Debugging**

When something goes wrong, work **systematically** rather than changing code at random.

1. **Reproduce** the problem with the smallest possible input — ideally as a failing test.
2. **Read the traceback** from the bottom up: exception type, message, then the line and call chain.
3. **Inspect state**: print or log the values just before the failure, or pause with the debugger.
4. **Form a hypothesis**, change **one** thing, and re-run.
5. **Keep the test** so the bug cannot return unnoticed (a *regression test*).

### **Tools**

| Tool                    | How                                                              |
| ----------------------- | ---------------------------------------------------------------- |
| `print()` / f-strings   | quick checks: `print(f"{order.status=}")`                        |
| `logging`               | permanent, configurable messages with levels (`DEBUG`, `INFO`, `WARNING`) |
| `breakpoint()`          | pauses execution and opens the `pdb` debugger at that line       |
| `%debug` (Jupyter)      | opens the debugger *after* an exception, at the failing line     |
| IDE debugger            | breakpoints, step over/into, watch variables (VS Code, PyCharm)  |

```python
def total(self):
    breakpoint()          # execution stops here; type 'p self._lines', 'n' (next), 'c' (continue)
    return sum(item.price * qty for item, qty in self._lines)
```

The `f"{expr=}"` form prints both the expression and its value, which makes quick debugging output self-explanatory:

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
log = logging.getLogger("campuseats")

order = canteen.place_order("Carlo", {"RICE": 1})
log.info("placed order #%s for %s", order.order_id, order.customer)
print(f"{order.total()=}, {order.status.name=}")

## **Version Control in a Team**

### **A branch per task**

```bash
git switch -c feature/place-order        # 1. branch from main
# ... write tests and code ...
python -m pytest                          # 2. make sure tests pass
git add src/campuseats/orders.py tests/test_orders.py
git commit -m "Add Order.total() and line validation"   # 3. small, focused commits
git push -u origin feature/place-order    # 4. open a pull request on GitHub
```

A teammate **reviews** the pull request; after approval and passing tests, it is **merged** into `main`.

```{mermaid}
gitGraph
    commit id: "skeletons"
    branch place-order
    checkout place-order
    commit id: "Order + tests"
    commit id: "Canteen.place_order"
    checkout main
    branch report
    checkout report
    commit id: "SalesReport"
    checkout main
    merge place-order
    merge report
```

### **Good commit messages**

| Weak               | Better                                              |
| ------------------ | --------------------------------------------------- |
| `update`           | `Validate item availability in Order.add_line()`    |
| `fix bug`          | `Fix total() ignoring quantity for repeated items`  |
| `final na talaga`  | `Add sales report CSV export and tests`             |

### **Merge conflicts**

A conflict happens when two branches change the **same lines**. Git marks them in the file:

```text
<<<<<<< HEAD
        return sum(item.price * qty for item, qty in self._lines)
=======
        return round(sum(item.price * qty for item, qty in self._lines), 2)
>>>>>>> feature/rounding
```

Decide what the correct code is, delete the markers, run the tests, then `git add` and `git commit`. Small, frequent merges keep conflicts small.

### **Code review checklist**

* Does the change match the **design** (responsibilities, relationships)?
* Are inputs **validated**, and are errors raised with clear messages?
* Are there **tests** for the new behavior, and do all tests pass?
* Are names clear and is the code readable (PEP 8)?
* Is anything private (passwords, real student data) accidentally committed?

## **Summary**

| Practice                    | Why                                                               |
| --------------------------- | ----------------------------------------------------------------- |
| skeletons & vertical slices | turn the design into working features early                       |
| dependency injection        | loose coupling; easy testing                                      |
| layered error handling      | models raise, the interface reports; objects stay consistent      |
| tests per acceptance criterion | proof that requirements are met                                |
| systematic debugging        | reproduce, read, inspect, hypothesize, keep a regression test     |
| branches, reviews, small commits | safe collaboration and a readable history                    |

## **References**

Beck, K. (2002). *Test-driven development: By example*. Addison-Wesley.

Chacon, S., & Straub, B. (2014). *Pro Git* (2nd ed.). Apress. https://git-scm.com/book/en/v2

Martin, R. C. (2008). *Clean code: A handbook of agile software craftsmanship*. Prentice Hall.

pytest-dev. (2026). *pytest documentation*. https://docs.pytest.org/

Python Software Foundation. (2026). *logging — Logging facility for Python*. https://docs.python.org/3/library/logging.html

Python Software Foundation. (2026). *The Python tutorial: 8. Errors and exceptions*. https://docs.python.org/3/tutorial/errors.html

Python Software Foundation. (2026). *unittest — Unit testing framework*. https://docs.python.org/3/library/unittest.html

Thomas, D., & Hunt, A. (2019). *The pragmatic programmer: Your journey to mastery* (20th anniversary ed.). Addison-Wesley.